In [5]:
# C2 - Install package from Git if needed
# Checks if the package is installed; skips install when present (no version/ref check).
# Run this cell, then restart the kernel if packages/imports changed.

import importlib.util
import subprocess
import sys

REPO_URL = 'https://github.com/manicko/mko_get_mediascope_data.git'
GIT_REF = 'master'  # selected ref (pinned commit SHA, tag, or branch)

# Optional non-default override examples (uncomment to use):
# GIT_REF = "0743c3d21ed5ac6291ab0a96dfab26db0664c05f"  # pinned delivery commit
# GIT_REF = "vX.Y.Z"  # use only if a tag exists on the remote

FORCE_REINSTALL = False  # set True to reinstall even if the package is already installed

package_name = "mko_get_mediascope_data"
install_spec = f"git+{REPO_URL}@{GIT_REF}"
print(f"Target: {install_spec}")

installed = importlib.util.find_spec(package_name) is not None
if FORCE_REINSTALL:
    print("FORCE_REINSTALL=True -> installation required.")
    do_install = True
elif installed:
    print(f"Package '{package_name}' is already installed; skipping installation.")
    print("To update or change ref, set FORCE_REINSTALL=True (or reinstall manually), then restart the kernel.")
    do_install = False
else:
    print(f"Package '{package_name}' is not installed; installation required.")
    do_install = True

if do_install:
    if "get_ipython" in globals():
        get_ipython().run_line_magic("pip", f'install --upgrade "{install_spec}"')
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", install_spec])
    print("Install finished. Restart the kernel before continuing if packages/imports changed.")
else:
    print("No install needed.")


Target: git+https://github.com/manicko/mko_get_mediascope_data.git@master
Package 'mko_get_mediascope_data' is already installed; skipping installation.
To update or change ref, set FORCE_REINSTALL=True (or reinstall manually), then restart the kernel.
No install needed.


In [6]:
# C3 - Imports and environment info (sanitized)

import json
import platform
import shutil
import sys
from pathlib import Path

import json5
import platformdirs

import mko_get_mediascope_data
from mko_get_mediascope_data.core.paths import APP_DIR as PACKAGE_APP_DIR, APP_NAME, AppPaths, PathResolver
from mko_get_mediascope_data.core.progress import NullProgressReporter
from mko_get_mediascope_data.core.services import AppService

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Package version:", getattr(mko_get_mediascope_data, "__version__", "unknown"))
print("Package location (sanitized):", Path(mko_get_mediascope_data.__file__).resolve().parent)
print("Bundled app dir:", PACKAGE_APP_DIR)

Python: 3.14.7
Platform: Windows-10-10.0.17763-SP0
Package version: 0.0.1.dev229+g7215341db
Package location (sanitized): C:\Users\MatveichevN\AppData\Roaming\Python\Python314\site-packages\mko_get_mediascope_data
Bundled app dir: C:\Users\MatveichevN\AppData\Roaming\Python\Python314\site-packages\mko_get_mediascope_data


In [7]:
# C4 - Configurable folder/credential variables (CR-3, CR-7)
# Set variables in this cell. Paths are portable by default; Windows examples are comments only.

# Portable workspace root (override as needed).
WORKSPACE_ROOT = Path.cwd() / ".mediascope_notebook_workspace"
SETTINGS_WORKSPACE = WORKSPACE_ROOT / "settings"   # settings/config workspace
REPORTS_DIR = SETTINGS_WORKSPACE / "reports"       # reports source path (workspace-local)
EXPORT_DIR = WORKSPACE_ROOT / "output"             # output/export

# OQ-2 default example report (workspace-local after bootstrap). Override allowed.
REPORT_FILENAME = "nat_tv_brands_theiss.json5"
REPORT_PATH = REPORTS_DIR / REPORT_FILENAME
# Example override to user config/external path (Windows example, comment only):
# REPORT_PATH = Path(r"%LOCALAPPDATA%\mko_get_mediascope_data\mko_get_mediascope_data\settings\reports\nat_tv_brands_theiss.json5")

# Credentials: external/user-managed file OUTSIDE notebook content. Never embed secrets.
# Portable user-config default:
CREDENTIALS_PATH = Path(platformdirs.user_config_dir(APP_NAME)) / "settings" / "connections" / "mediascope.json"
# Windows example (comment only):
# import os
# CREDENTIALS_PATH = Path(os.environ["LOCALAPPDATA"]) / "mko_get_mediascope_data" / "mko_get_mediascope_data" / "settings" / "connections" / "mediascope.json"

OVERWRITE_WORKSPACE_TEMPLATES = False   # non-destructive default
COPY_CREDENTIALS_INTO_WORKSPACE = False  # explicit opt-in; never overwrites existing workspace creds
DRY_RUN_BOOTSTRAP_ONLY = False           # setup check only - NOT CR-5 success

BUNDLED_SETTINGS_DIR = PACKAGE_APP_DIR / "settings"

REPO_URL@GIT_REF: https://github.com/manicko/mko_get_mediascope_data.git@master
WORKSPACE_ROOT: C:\py_exp\mko_get_mediascope_data\examples\.mediascope_notebook_workspace
SETTINGS_WORKSPACE: C:\py_exp\mko_get_mediascope_data\examples\.mediascope_notebook_workspace\settings
REPORTS_DIR: C:\py_exp\mko_get_mediascope_data\examples\.mediascope_notebook_workspace\settings\reports
EXPORT_DIR: C:\py_exp\mko_get_mediascope_data\examples\.mediascope_notebook_workspace\output
REPORT_PATH: C:\py_exp\mko_get_mediascope_data\examples\.mediascope_notebook_workspace\settings\reports\nat_tv_brands_theiss.json5
CREDENTIALS_PATH: <external, user-managed; path configured>
COPY_CREDENTIALS_INTO_WORKSPACE: False
DRY_RUN_BOOTSTRAP_ONLY: False


In [ ]:
# C6 - Workspace bootstrap (must run before C7/AppService construction)
# CR-7: excludes credential files from blanket template copy; never overwrites existing
# workspace creds; never silently promotes bundled placeholder.
# CR-3: sets workspace app_config export_folder to absolute EXPORT_DIR preserving other keys.

CREDENTIAL_REL = Path("connections") / "mediascope.json"
WORKSPACE_CREDENTIAL = SETTINGS_WORKSPACE / CREDENTIAL_REL


def update_export_folder(app_config_path: Path, export_dir: Path) -> None:
    data = json5.loads(app_config_path.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise TypeError(f"Expected dict in {app_config_path}")
    data["export_folder"] = str(export_dir.resolve())
    app_config_path.write_text(json.dumps(data, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


# 1) Copy non-credential templates (workspace-local settings tree)
copy_non_credential_templates(
    BUNDLED_SETTINGS_DIR, SETTINGS_WORKSPACE, overwrite=OVERWRITE_WORKSPACE_TEMPLATES
)

# 2) Ensure selected report exists in workspace reports dir (copy bundled default if missing)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
workspace_report = REPORTS_DIR / REPORT_FILENAME
if not workspace_report.exists() and bundled_report.exists():
    shutil.copy2(bundled_report, workspace_report)

# 3) Safe credential handling (no bundled placeholder promotion)
cred_ws = ensure_workspace_credentials()

# 4) CR-3 export mapping: absolute EXPORT_DIR before AppService construction
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
update_export_folder(SETTINGS_WORKSPACE / "app_config.json5", EXPORT_DIR)

# 5) Minimal completeness checks
required = [
    SETTINGS_WORKSPACE / "app_config.json5",
    SETTINGS_WORKSPACE / "log_config.json5",
    SETTINGS_WORKSPACE / "defaults" / "tv" / "crosstab.json5",
    REPORT_PATH,
]
missing = [str(p) for p in required if not p.exists()]
print("Credential workspace action:", cred_ws["action"])
print("Workspace credentials present:", WORKSPACE_CREDENTIAL.exists())
print("Export folder mapped to absolute:", EXPORT_DIR.resolve())
print("Missing required paths:", missing if missing else "none")
if missing:
    print("Bootstrap incomplete: fix missing paths before constructing AppService.")

In [ ]:
# C7 - Construct DI AppService (Alt A). Run C6 first.
# Side effect: AppService.prepare_app() applies global logging.config.dictConfig().

app_paths = AppPaths(user_dir=WORKSPACE_ROOT, app_dir=PACKAGE_APP_DIR, app_name=APP_NAME)
resolver = PathResolver(WORKSPACE_ROOT)
service = AppService(app_paths, resolver)

print("Service constructed with workspace-local DI.")
print("Resolved export folder (from service):", service.export_folder)
print("Note: public singleton wrappers and CLI init/list-reports are not used for this Alt A path.")

In [ ]:
# C8 - Run report (authorized real E2E) or report blocked state
# Requires native `await` (Jupyter/IPython active loop). No event-loop nesting packages.

if blocked_reason:
    print("RUN STATUS: BLOCKED - no success claim.")
    print("Reason:", blocked_reason)
    export_paths = []
else:
    try:
        export_paths = await service.run_report(Path(REPORT_PATH), progress=NullProgressReporter())
        # run_report returns export target paths; individual entries may be skipped (warned).
        print("RUN STATUS: COMPLETED (check warnings/output verification; entries may be skipped).")
        print("Returned export target paths:", [str(p) for p in export_paths])
    except Exception as exc:
        export_paths = []
        # Do not print secrets/auth payloads; show exception type and non-sensitive message.
        print("RUN STATUS: FAILED - no success claim.")
        print(f"{type(exc).__name__}: {exc}")

In [ ]:
# C9 - Verify/list export files (no sensitive content)
# Do not claim success from path existence alone; run_report may skip entries.

def list_files(root: Path):
    if not root.exists():
        return []
    return sorted([p for p in root.rglob("*") if p.is_file()])


files = list_files(EXPORT_DIR)
print("EXPORT_DIR:", EXPORT_DIR)
print("File count:", len(files))
for p in files[:200]:
    try:
        size = p.stat().st_size
    except OSError:
        size = "?"
    print(f"- {p.relative_to(EXPORT_DIR)} ({size} bytes)")
if len(files) > 200:
    print(f"... ({len(files) - 200} more files omitted)")
if blocked_reason:
    print("Verification note: run was BLOCKED; missing outputs are expected.")